# The floor is representation — seen directly

Notebooks 03–04 ruled out instability, reward variance, training budget, and coverage. What's left is
a one-directional **over-doubling bias** in a handful of boundary cells. A bias in specific cells is a
statement about how the network *represents* the decision map — so this notebook looks straight at
that representation, for one trained run, and shows the errors live exactly where the geometry
predicts. (Notebook 06 then turns the encoding into a dial and confirms the *cause*.)

Method: reload the network's weights and read the **penultimate-layer activation** for every decision
cell — the learned features the Q-head sees — then project to 2D. The input is already 2–3
interpretable dimensions, so this isn't re-deriving the basic-strategy heatmap; it shows the *learned
geometry* and, the point, **where the genuine errors sit**.

In [ ]:
import sys
from pathlib import Path
import numpy as np
sys.path.insert(0, '.')
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'blackjack_rl').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from runs import load_runs
from blackjack_rl.dqn.embedding import load_agent, cell_embeddings

df = load_runs(); dqn = df[df.has_model & (df.method == 'dqn')]
# one canonical, well-trained run: the long scalar flat-then-decay (the smooth encoding, where the
# boundary bleed is clearest). Edit to inspect a different run.
RUN = dqn[(dqn.encoding == 'scalar') & (dqn.lr_hold_until > 0) & (dqn.episodes == 5_000_000)].iloc[0]
ce = cell_embeddings(load_agent(RUN.path))
X = np.array(ce.embeddings); cells = ce.cells
print('run        :', RUN.run, f'({RUN.encoding}, {RUN.episodes//1000}k, agreement {RUN.agreement:.1%})')
print('embeddings :', X.shape, '(one penultimate-layer vector per decision cell)')

## PCA — three views of the same projection

By chosen **action** (does it carve coherent regions?), by diff **category** (the key panel — do the
errors cluster?), and by **decision margin** `Q(best) − Q(2nd)` (the net's own confidence; low margin
traces the boundaries).

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

fit = PCA(n_components=2).fit(X); Y = fit.transform(X); evr = fit.explained_variance_ratio_
ACT = {'hit': '#378ADD', 'stand': '#1D9E75', 'double': '#D85A30'}
CAT = {'agree': '#CBCBC4', 'near_equal_ev': '#EF9F27', 'genuine_disagreement': '#E24B4A'}
def scat(ax, pts, labels, colors, title):
    for lab in dict.fromkeys(labels):
        m = [i for i, l in enumerate(labels) if l == lab]
        ax.scatter(pts[m, 0], pts[m, 1], s=24, c=colors.get(lab, '#888'), label=str(lab), edgecolor='none', alpha=0.85)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.legend(fontsize=8, framealpha=0.9)
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
scat(ax[0], Y, [c['action'] for c in cells], ACT, 'by chosen action')
scat(ax[1], Y, [c['category'] for c in cells], CAT, 'by diff category')
sc = ax[2].scatter(Y[:, 0], Y[:, 1], s=24, c=[c['q_margin'] for c in cells], cmap='viridis')
ax[2].set_title('by decision margin  Q(best)-Q(2nd)'); ax[2].set_xticks([]); ax[2].set_yticks([])
fig.colorbar(sc, ax=ax[2], fraction=0.046, pad=0.04)
fig.suptitle(f'PCA of last-layer embedding — {RUN.run}  (PC1 {evr[0]:.0%}, PC2 {evr[1]:.0%} var)', y=1.02)
plt.tight_layout(); plt.show()

## Double is the seam, and the errors live on it

The action panel shows a **sensible** map — the representation cleanly separates hit / stand / double,
with double sitting *between* hit and stand (doubling is the in-between decision). The category panel
shows the red errors are **not scattered** — they hug the edge of the double region, where it meets
hit and stand. And the margin panel shows that edge is exactly where the net's confidence collapses.
Quantified below: the genuine-disagreement cells have a far smaller decision margin than the cells the
net gets right, and the errors are almost all over-doubles (double bleeding past its boundary).

In [ ]:
from sklearn.metrics import silhouette_score
act = [c['action'] for c in cells]; code = {a: i for i, a in enumerate(dict.fromkeys(act))}
print(f"action silhouette (full embedding space): {silhouette_score(X, [code[a] for a in act]):+.3f}  (>0 = coherent action regions)")
m_agree = [c['q_margin'] for c in cells if c['category'] == 'agree']
m_dis = [c['q_margin'] for c in cells if c['category'] == 'genuine_disagreement']
print('mean decision margin  Q(best)-Q(2nd):')
print(f'   cells it gets RIGHT (agree)    : {np.mean(m_agree):.2f}')
print(f'   cells it gets WRONG (disagree) : {np.mean(m_dis):.2f}   <- on the razor-edge of the boundary')
over = [c for c in cells if c['category'] == 'genuine_disagreement' and c['action'] == 'double' and c['basic_action'] != 'double']
print(f'\nof the {len(m_dis)} genuine errors, {len(over)} are over-doubles (double bleeding past its boundary).')

## t-SNE — a nonlinear cross-check (secondary)

At N=240 the layout is perplexity-sensitive, so treat it as confirmation of the PCA story, not truth
in its own right. It separates the actions into clearer islands; again the red errors sit on the
boundaries *between* the islands, not inside them.

In [ ]:
from sklearn.manifold import TSNE
T = TSNE(n_components=2, perplexity=30, init='pca', random_state=0).fit_transform(X)
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
scat(ax[0], T, [c['action'] for c in cells], ACT, 't-SNE by action')
scat(ax[1], T, [c['category'] for c in cells], CAT, 't-SNE by category')
fig.suptitle(f't-SNE (secondary, perplexity=30, N={len(cells)})', y=1.02)
plt.tight_layout(); plt.show()

## Conclusion — the function-approximation floor, drawn out

The errors are not where the net is *wrong* so much as where it is *uncertain* — the low-margin seam
between action regions, which is exactly where the true EV difference between doubling and not is
razor-thin. A continuous representation places adjacent cells next to each other and **interpolates**
between them, so a boundary that should be a sharp step becomes a soft ramp, and the cells sitting on
the ramp fall to the wrong side. That is the floor: not instability, not under-training — the geometry
of a smooth approximator on a problem with hard, exact boundaries.

This closes the loop with notebook 01. A lookup table has no "between": every cell is independent, so
its boundary is a wall and it pays nothing for sharpness. The network's boundary is a gradient, and
the over-doubles are the cells caught on the slope. If that diagnosis is right, then making the
representation *less* smooth should sharpen the boundary and shrink the over-doubles — which is exactly
the experiment in **notebook 06**.